In [3]:
!pip install -q crewai crewai-tools

In [4]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from crewai import LLM
llm = LLM(
    model="gemini/gemini-flash-lite-latest",
    temperature=0.2)

In [5]:
leads = [
    {"name": "Arjun Mehta",
     "website": "https://www.zoho.com",
     "message": "We need a CRM for our 50-person sales team. Budget approved, "
                "want to start next month. Can we get a demo this week?"},
    {"name": "Priya Nair",
     "website": "https://www.freshworks.com",
     "message": "Exploring options for later this year. Could you send some "
                "info about your pricing?"},
    {"name": "Rahul",
     "website": "https://example.com",
     "message": "just checking what this is"},
]

In [6]:
from crewai import Agent, Task, Crew
from crewai_tools import ScrapeWebsiteTool

scrape_tool = ScrapeWebsiteTool()

async def run_crew(role, goal, instructions, text, tools=None):
    agent = Agent(role=role, goal=goal,
                  backstory="You are an experienced " + role + ".",
                  tools=tools or [],
                  llm=llm, verbose=False)
    task = Task(description=instructions + "\n\nDetails:\n" + text,
                expected_output="A short, clear response.",
                agent=agent)
    crew = Crew(agents=[agent], tasks=[task], verbose=False)
    result = await crew.kickoff_async()
    return result.raw

In [7]:
from crewai.flow.flow import Flow, start, listen, router, or_
import csv

nurture_list = []

class LeadFlow(Flow):

    @start()
    def capture_lead(self):
        print("\nNew lead:", self.state["name"])

    @listen(capture_lead)
    async def enrich_lead(self):
        self.state["company_info"] = await run_crew(
            role="Business Researcher",
            goal="Understand what a company does from its website",
            instructions="Read this website using your tool and describe in 3 lines "
                         "what the company does. If the website cannot be read, "
                         "reply 'Website info unavailable'.",
            text=self.state["website"],
            tools=[scrape_tool],
        )

    @router(enrich_lead)
    def score_lead(self):
        answer = llm.call(
            "Label this sales lead as one word:\n"
            "- hot: clear need, budget or timeline, wants to talk soon\n"
            "- warm: interested but no urgency or budget yet\n"
            "- cold: no real interest or too vague\n"
            "Reply with one word only.\n\n"
            "Message: " + self.state["message"] + "\n"
            "Company: " + self.state["company_info"]
        )
        label = answer.strip().lower()
        if label not in ["hot", "warm", "cold"]:
            label = "warm"
        self.state["label"] = label
        print("Label:", label)
        return label

    @listen("hot")
    async def send_outreach(self):
        email = await run_crew(
            role="Sales Executive",
            goal="Write personal sales emails that get replies",
            instructions="Write a short, personal email (5 lines) to this lead. "
                         "Mention their need and offer a demo this week.",
            text="Name: " + self.state["name"] + "\nMessage: " + self.state["message"]
                 + "\nCompany: " + self.state["company_info"],
        )
        self.state["action"] = "Outreach email drafted"
        print("Email:\n" + email)

    @listen("warm")
    def add_to_nurture(self):
        nurture_list.append(self.state["name"])
        self.state["action"] = "Added to weekly tips list"

    @listen("cold")
    def archive_lead(self):
        self.state["action"] = "Archived"

    @listen(or_(send_outreach, add_to_nurture, archive_lead))
    def log_lead(self):
        with open("leads.csv", "a", newline="") as f:
            csv.writer(f).writerow([self.state["name"], self.state["website"],
                                    self.state["label"], self.state["action"]])
        print("Logged:", self.state["action"])

In [8]:
if os.path.exists("leads.csv"):
    os.remove("leads.csv")
nurture_list.clear()

for lead in leads:
    flow = LeadFlow()
    await flow.kickoff_async(inputs=lead)

print("\nWeekly tips list:", nurture_list)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: LeadFlow                                                                                                 │
│  ID: 1372d17d-9c60-4a22-a742-11e5bce35213                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: LeadFlow                                                                                                 │
│  ID: 1372d17d-9c60-4a22-a742-11e5bce35213                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: capture_lead                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


New lead: Arjun Mehta


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: enrich_lead                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: log_lead                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: archive_lead                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: archive_lead                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: LeadFlow                                                                                                 │
│  ID: 84f0e542-1d9b-44fe-b359-ff9579a131f7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Weekly tips list: ['Priya Nair']


In [9]:
import pandas as pd
pd.read_csv("leads.csv", names=["name", "website", "label", "action"])

,name,website,label,action
0,Arjun Mehta,https://www.zoho.com,hot,Outreach email drafted
1,Priya Nair,https://www.freshworks.com,warm,Added to weekly tips list
2,Rahul,https://example.com,cold,Archived
